# RUKOPYS — Kaggle submission (Mavka team)

**Pipeline**
1. **Detection:** DocLayout-YOLO (`best.pt`, class-agnostic boxes), conf 0.25, imgsz 1024, as in `notebooks/06`.
2. **Classification:** Qwen3-VL-4B + LoRA `step_000350`, using the exported `RegionClassifier` code from notebook 07.
3. **Text:**
   - `handwritten` / `printed` / `annotation` → TrOCR-large (our fine-tune, val CER 0.0905);
   - `formula` / `table` → Qwen3-VL-8B + LoRA `best_adapter` (the team's `infer_formula_table.py`, unchanged);
   - `image` / `graph` → empty text.
4. **Export:** CSV for Kaggle, plus an approximate local score on the HF test labels (not the official metric; for sanity only, never for tuning).

**Before running**
- **Shortcuts:** in Google Drive → *Shared with me*, right-click each shared folder → *Organize* → *Add shortcut* → *My Drive*. Needed: `rukopys_ft_qwen3vl8b_v1` and `rukopys_qwen_classifier_t4_224_v1`.
- **Detector:** the detector weights `best.pt` must be somewhere in your Drive. Cell 2 searches for them by checksum, or set `DETECTOR_WEIGHTS` yourself.
- **Runtime:** **L4 GPU** recommended (native bf16 for the 8B model). On a T4 it works, but the 8B stage is much slower.

**Order**
1. Cell 1 → *Runtime → Restart session* → cells 2–9 with `SMOKE = True` (3 pages; ~20–30 min the first time, mostly downloading the 4B and 8B base models). Check the printed regions.
2. Set `SMOKE = False` in cell 2 and run cells 2–9 again.

Every stage saves its results to Drive, so after a disconnect, rerunning the cells continues where it stopped.

## 1. Packages (then Runtime → Restart session)

In [7]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [8]:
!pip list 2>/dev/null | grep -i -E "doclayout|^transformers |peft|bitsandbytes"

peft                                  0.21.2
transformers                          5.22.1


In [9]:
!pip list 2>/dev/null | grep -i -E "doclayout|^transformers |bitsandbytes|^peft "

peft                                  0.21.2
transformers                          5.22.1


In [10]:
!pip -q install "transformers>=4.57,<5" "accelerate>=1.10,<2" "peft>=0.17,<1" "bitsandbytes>=0.46.1" \
               "huggingface_hub>=0.34,<1" doclayout-yolo rapidfuzz
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Red pip dependency warnings about gradio/diffusers are harmless here.
# NOW: Runtime -> Restart session, then continue with cell 2.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [11]:
!python -c "import huggingface_hub, transformers, doclayout_yolo, peft, bitsandbytes; print('OK', transformers.__version__, huggingface_hub.__version__)"

I0000 00:00:1795754160.623044    3663 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
OK 4.57.6 0.36.2


## 2. Settings, paths, checks

In [2]:
SMOKE = False          # True: 3 pages to check the pipeline. Then False for all 385 pages.
CONF, IMGSZ = 0.25, 1024
DETECTOR_WEIGHTS = None   # e.g. '/content/drive/MyDrive/some_folder/best.pt'; None = search Drive by checksum
EXPECTED_DET_SHA = '2c4f64810fe020deb7d19e13513d8c588cdbe73a7f762503125e35eb9a110b65'   # from notebooks/06

import os, gc, re, json, csv, math, hashlib, importlib.util
from pathlib import Path
import torch
from PIL import Image
Image.MAX_IMAGE_PIXELS = None     # some scans are >100 MP
from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive')
OCR_DIR = DRIVE / 'rukopys_ft_qwen3vl8b_v1'
OCR_ADAPTER, OCR_INFER = OCR_DIR / 'best_adapter', OCR_DIR / 'infer_formula_table.py'
CLS_CKPT = DRIVE / 'rukopys_qwen_classifier_t4_224_v1' / 'checkpoints' / 'step_000350'
TROCR_DIR = DRIVE / 'rukopys_trocr' / 'trocr_large_gold' / 'final'
OUT = DRIVE / 'rukopys_submission_v1' / ('smoke' if SMOKE else 'full')
OUT.mkdir(parents=True, exist_ok=True)
DATA = Path('/content/rukopys_test')

problems = []
for label, p in [('8B adapter', OCR_ADAPTER / 'adapter_config.json'), ('8B inference module', OCR_INFER),
                 ('Classifier adapter', CLS_CKPT / 'adapter_config.json'), ('TrOCR model', TROCR_DIR / 'config.json')]:
    ok = p.exists(); print(f'{label:22s}', 'FOUND  ' if ok else 'MISSING', p)
    if not ok: problems.append(label)
if problems:
    raise FileNotFoundError(f'Missing: {problems}. For shared folders: Drive -> Shared with me -> right-click -> '
                            'Organize -> Add shortcut -> My Drive, then rerun this cell.')

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()

if DETECTOR_WEIGHTS:
    DET_W = Path(DETECTOR_WEIGHTS); assert DET_W.is_file(), f'Not found: {DET_W}'
    if sha256(DET_W) != EXPECTED_DET_SHA:
        print('WARNING: these weights differ from the ones used for the 0.578 submission (checksum mismatch).')
else:
    cands = [p for pat in ['*.pt', '*/*.pt', '*/*/*.pt', '*/*/*/*.pt', '*/*/*/*/*.pt'] for p in DRIVE.glob(pat)
             if p.name != 'training_state.pt' and p.stat().st_size < 300e6]
    cands.sort(key=lambda p: 'best' not in p.name)          # check files named best*.pt first
    DET_W = next((p for p in cands if sha256(p) == EXPECTED_DET_SHA), None)
    if DET_W is None:
        print('.pt files found in Drive:', *cands, sep='\n  ')
        raise FileNotFoundError('Detector best.pt with the expected checksum not found. Ask Viktoriya for best.pt, '
                                'put it in Drive and set DETECTOR_WEIGHTS to its path.')
print('Detector weights:', DET_W)

gpu = torch.cuda.get_device_name(0); VRAM = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'GPU: {gpu} ({VRAM:.0f} GiB) | SMOKE={SMOKE} | results -> {OUT}')
if 'T4' in gpu: print('NOTE: T4 has no native bf16; the 8B stage will be slow. L4 is recommended.')

def load_json(p, default):
    return json.loads(Path(p).read_text(encoding='utf-8')) if Path(p).exists() else default
def save_json(p, obj):
    tmp = Path(str(p) + '.tmp'); tmp.write_text(json.dumps(obj, ensure_ascii=False), encoding='utf-8'); tmp.replace(p)
def free_gpu():
    gc.collect(); torch.cuda.empty_cache()

Mounted at /content/drive
8B adapter             FOUND   /content/drive/MyDrive/rukopys_ft_qwen3vl8b_v1/best_adapter/adapter_config.json
8B inference module    FOUND   /content/drive/MyDrive/rukopys_ft_qwen3vl8b_v1/infer_formula_table.py
Classifier adapter     FOUND   /content/drive/MyDrive/rukopys_qwen_classifier_t4_224_v1/checkpoints/step_000350/adapter_config.json
TrOCR model            FOUND   /content/drive/MyDrive/rukopys_trocr/trocr_large_gold/final/config.json
Detector weights: /content/drive/MyDrive/best.pt
GPU: Tesla T4 (15 GiB) | SMOKE=False | results -> /content/drive/MyDrive/rukopys_submission_v1/full
NOTE: T4 has no native bf16; the 8B stage will be slow. L4 is recommended.


## 3. Test images (HF, ~1 GB) and the official image list

In [3]:
from huggingface_hub import snapshot_download, HfApi
commits = HfApi().list_repo_commits('UkrainianCatholicUniversity/rukopys', repo_type='dataset')
REV = next(c.commit_id for c in commits if c.title.startswith('v2.7:'))
DATA = Path('/content/rukopys_test_v27')
print('dataset revision:', REV)
snapshot_download('UkrainianCatholicUniversity/rukopys', repo_type='dataset', revision=REV, local_dir=str(DATA),
                  allow_patterns=['test/*', 'sample_submission.csv'], max_workers=8)

meta = {}
for line in (DATA / 'test' / 'metadata.jsonl').read_text(encoding='utf-8').splitlines():
    r = json.loads(line); meta[Path(r['file_name']).name] = r
sample = DATA / 'sample_submission.csv'
if sample.exists():
    with open(sample, encoding='utf-8-sig', newline='') as f:
        rd = csv.DictReader(f); assert rd.fieldnames == ['image', 'regions'], rd.fieldnames
        EXPECTED = [row['image'] for row in rd]
else:
    print('WARNING: sample_submission.csv not found; using metadata order.')
    EXPECTED = list(meta)
assert len(EXPECTED) == 385 and len(set(EXPECTED)) == 385, len(EXPECTED)
missing = [n for n in EXPECTED if not (DATA / 'test' / 'images' / n).is_file()]
assert not missing, f'{len(missing)} test images missing, e.g. {missing[:3]} - rerun this cell'

PAGES = []
for n in EXPECTED:
    path = DATA / 'test' / 'images' / n
    with Image.open(path) as im: w, h = im.size
    PAGES.append({'image': n, 'path': str(path), 'width': w, 'height': h})
RUN_PAGES = PAGES[:3] if SMOKE else PAGES
has_gt = sum(bool(meta[n].get('regions')) for n in EXPECTED)
print(f'{len(PAGES)} test pages ready | running on {len(RUN_PAGES)} | pages with HF labels: {has_gt}')

dataset revision: 8217bf9a5189f5ce919de99e32e5db7bab37fc17


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 387 files:   0%|          | 0/387 [00:00<?, ?it/s]

385 test pages ready | running on 385 | pages with HF labels: 385


## 4. Stage 1 — detection

In [4]:
DET_FILE = OUT / 'detections.json'
dets = load_json(DET_FILE, {})
todo = [p for p in RUN_PAGES if p['image'] not in dets]
if todo:
    from doclayout_yolo import YOLOv10
    det = YOLOv10(str(DET_W))
    for k, p in enumerate(todo, 1):
        r = det.predict(p['path'], conf=CONF, imgsz=IMGSZ, device='cuda:0', verbose=False)[0]
        h, w = r.orig_shape
        if (w, h) != (p['width'], p['height']):
            raise ValueError(f"{p['image']}: detector sees {(w, h)}, PIL sees {(p['width'], p['height'])} (EXIF rotation?)")
        boxes = []
        for b, s in zip(r.boxes.xyxy.cpu().tolist(), r.boxes.conf.cpu().tolist()):
            x1, y1, x2, y2 = max(0., b[0]), max(0., b[1]), min(float(w), b[2]), min(float(h), b[3])
            if x2 - x1 >= 2 and y2 - y1 >= 2:
                boxes.append({'bbox': [round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)], 'conf': round(s, 4)})
        dets[p['image']] = boxes
        if k % 25 == 0 or k == len(todo):
            save_json(DET_FILE, dets); print(f'detected {len(dets)}/{len(RUN_PAGES)} pages')
    del det; free_gpu()
n_boxes = sum(len(dets[p['image']]) for p in RUN_PAGES)
print(f'Detection done: {n_boxes} regions on {len(RUN_PAGES)} pages ({n_boxes / len(RUN_PAGES):.1f} per page; GT test average ~21.5)')

Detection done: 8961 regions on 385 pages (23.3 per page; GT test average ~21.5)


## 5. Stage 2 — classification (Qwen3-VL-4B + LoRA)

In [5]:
CLASSIFIER_SRC = '"""Qwen3-VL-4B 7-label RUKOPYS classifier. The preprocessing and prompt\nmatch RUKOPYS_01_Qwen_Classification_T4_OOM_Fixed.ipynb.\nRequires CUDA, transformers, peft, bitsandbytes, PIL and torch.\n"""\nimport torch\nfrom PIL import Image, ImageOps\nfrom transformers import AutoProcessor, Qwen3VLForConditionalGeneration, BitsAndBytesConfig\nfrom peft import PeftModel\n\nCLASSES=[\'handwritten\',\'printed\',\'formula\',\'table\',\'annotation\',\'image\',\'graph\']\nPROMPT=(\'Classify the main content of this document crop. Return exactly one label: \'\n    \'handwritten, printed, formula, table, annotation, image, graph. \'\n    \'Use annotation for marginal notes, signatures or correction marks; formula for mathematical expressions; \'\n    \'table for tabular content; graph for plotted charts; image for non-text illustrations. No explanation.\')\nBASE=\'Qwen/Qwen3-VL-4B-Instruct\'\n\ndef fit_image(image):\n    image=ImageOps.exif_transpose(image.convert(\'RGB\'))\n    image.thumbnail((224,224),Image.Resampling.LANCZOS)\n    canvas=Image.new(\'RGB\',(224,224),\'white\')\n    canvas.paste(image,((224-image.width)//2,(224-image.height)//2))\n    return canvas\n\nclass RegionClassifier:\n    def __init__(self,adapter_dir):\n        assert torch.cuda.is_available(),\'CUDA GPU required for 4-bit model\'\n        dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n        quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type=\'nf4\',\n            bnb_4bit_compute_dtype=dtype,bnb_4bit_use_double_quant=True)\n        self.processor=AutoProcessor.from_pretrained(BASE)\n        ip=self.processor.image_processor\n        # Newer Transformers exposes `image_processor.size` as SizeDict,\n        # not a built-in dict. Support both forms without changing the\n        # original training-time visual pixel budget.\n        pixel_limit = 224 * 224\n        size = ip.size\n        if isinstance(size, dict):\n            size[\'longest_edge\'] = pixel_limit\n            if size.get(\'shortest_edge\') is not None:\n                size[\'shortest_edge\'] = min(size[\'shortest_edge\'], pixel_limit)\n        elif hasattr(size, \'longest_edge\'):\n            size.longest_edge = pixel_limit\n            if getattr(size, \'shortest_edge\', None) is not None:\n                size.shortest_edge = min(size.shortest_edge, pixel_limit)\n        else:\n            raise TypeError(f\'Unsupported image processor size: {type(size).__name__}: {size!r}\')\n        if hasattr(ip, \'max_pixels\'):\n            ip.max_pixels = pixel_limit\n        base=Qwen3VLForConditionalGeneration.from_pretrained(BASE,quantization_config=quant,\n            device_map=\'auto\',torch_dtype=dtype)\n        self.model=PeftModel.from_pretrained(base,str(adapter_dir)).eval()\n\n    @torch.inference_mode()\n    def predict(self,crops):\n        result=[]\n        for crop in crops:\n            image=fit_image(crop)\n            msgs=[{\'role\':\'user\',\'content\':[{\'type\':\'image\',\'image\':image},\n                {\'type\':\'text\',\'text\':PROMPT}]}]\n            inputs=self.processor.apply_chat_template(msgs,tokenize=True,\n                add_generation_prompt=True,return_dict=True,return_tensors=\'pt\')\n            inputs.pop(\'token_type_ids\',None)\n            inputs=inputs.to(self.model.device)\n            gen=self.model.generate(**inputs,max_new_tokens=16,do_sample=False,\n                pad_token_id=self.processor.tokenizer.eos_token_id)\n            text=self.processor.batch_decode(gen[:,inputs[\'input_ids\'].shape[1]:],\n                skip_special_tokens=True)[0].strip().lower().strip(\' .`\\n\\t\')\n            result.append(text if text in CLASSES else \'__invalid__\')\n        return result\n'
Path('/content/infer_region_classification.py').write_text(CLASSIFIER_SRC, encoding='utf-8')
spec = importlib.util.spec_from_file_location('rk_cls', '/content/infer_region_classification.py')
rk_cls = importlib.util.module_from_spec(spec); spec.loader.exec_module(rk_cls)

def int_box(b, W, H):
    x1, y1, x2, y2 = b
    return (max(0, int(math.floor(x1))), max(0, int(math.floor(y1))), min(W, int(math.ceil(x2))), min(H, int(math.ceil(y2))))

CLS_FILE = OUT / 'classes.json'
classes = load_json(CLS_FILE, {})
todo = [p for p in RUN_PAGES if p['image'] not in classes]
if todo:
    clf = rk_cls.RegionClassifier(CLS_CKPT)
    proc, mdl = clf.processor, clf.model

    @torch.inference_mode()
    def predict_batched(crops):
        msgs = [[{'role': 'user', 'content': [{'type': 'image', 'image': rk_cls.fit_image(c)},
                                              {'type': 'text', 'text': rk_cls.PROMPT}]}] for c in crops]
        proc.tokenizer.padding_side = 'left'
        inp = proc.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_dict=True,
                                       return_tensors='pt', padding=True)
        inp.pop('token_type_ids', None); inp = inp.to(mdl.device)
        gen = mdl.generate(**inp, max_new_tokens=16, do_sample=False, pad_token_id=proc.tokenizer.eos_token_id)
        txt = proc.batch_decode(gen[:, inp['input_ids'].shape[1]:], skip_special_tokens=True)
        return [t if t in rk_cls.CLASSES else '__invalid__' for t in (s.strip().lower().strip(' .`\n\t') for s in txt)]

    # Batching is only used if it gives the same labels as the original one-by-one code.
    p0 = todo[0]
    with Image.open(p0['path']) as im:
        page = im.convert('RGB')
        test_crops = [page.crop(int_box(b['bbox'], *page.size)) for b in dets[p0['image']][:24]]
    seq = clf.predict(test_crops)
    try:
        bat = predict_batched(test_crops)
        agree = sum(a == b for a, b in zip(seq, bat)) / max(1, len(seq))
    except Exception as e:
        print('Batched classification failed:', type(e).__name__, e); agree = 0.0
    USE_BATCH = agree >= 0.95
    print(f'batched vs original agreement on {len(seq)} crops: {agree:.0%} -> using', 'batches of 16' if USE_BATCH else 'original one-by-one code')

    for k, p in enumerate(todo, 1):
        with Image.open(p['path']) as im:
            page = im.convert('RGB')
            crops = [page.crop(int_box(b['bbox'], *page.size)) for b in dets[p['image']]]
        labels = []
        for s in range(0, len(crops), 16):
            chunk = crops[s:s + 16]
            labels += predict_batched(chunk) if USE_BATCH else clf.predict(chunk)
        classes[p['image']] = labels
        if k % 10 == 0 or k == len(todo):
            save_json(CLS_FILE, classes); print(f'classified {len(classes)}/{len(RUN_PAGES)} pages')
    del clf, mdl, proc; free_gpu()

from collections import Counter
cnt = Counter(l for p in RUN_PAGES for l in classes[p['image']])
print('Predicted types:', dict(cnt.most_common()))
print('(HF test GT is ~84% handwritten. "__invalid__" labels become "handwritten" in the export.)')

Predicted types: {'handwritten': 5427, 'printed': 1509, 'formula': 1003, 'annotation': 939, 'table': 29, 'graph': 27, 'image': 27}
(HF test GT is ~84% handwritten. "__invalid__" labels become "handwritten" in the export.)


## 6. Stage 3 — text lines with TrOCR (handwritten / printed / annotation)

In [6]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
TEXT_TYPES = {'handwritten', 'printed', 'annotation', '__invalid__'}
MAX_LEN, PAD, MAX_W = 192, 4, 1536          # identical to training / cell 7 of the TrOCR notebook

def trocr_crop(page, bbox):                  # same crop as training: 4 px padding, downscale if wider than 1536
    W, H = page.size; x1, y1, x2, y2 = map(int, bbox)
    c = page.crop((max(0, x1 - PAD), max(0, y1 - PAD), min(W, x2 + PAD), min(H, y2 + PAD)))
    if c.width > MAX_W:
        c = c.resize((MAX_W, max(1, round(c.height * MAX_W / c.width))))
    return c

LOOP = re.compile(r'([^\W\d_]{1,6}?)\1{7,}')    # a letter block repeated 8+ times, e.g. 'blblblbl...'
def trim_loop(t):
    m = LOOP.search(t)
    return (t[:m.start()].rstrip(), True) if m else (t, False)

TXT_FILE = OUT / 'trocr.json'
trocr = load_json(TXT_FILE, {})
todo = [p for p in RUN_PAGES if p['image'] not in trocr]
loops = 0
if todo:
    tproc = TrOCRProcessor.from_pretrained(str(TROCR_DIR))
    tmodel = VisionEncoderDecoderModel.from_pretrained(str(TROCR_DIR)).cuda().eval().half()

    @torch.inference_mode()
    def recognize(images, bs=32):
        out = []
        for i in range(0, len(images), bs):
            pix = tproc(images=images[i:i + bs], return_tensors='pt').pixel_values.cuda().half()
            ids = tmodel.generate(pix, max_length=MAX_LEN, num_beams=1, no_repeat_ngram_size=0, use_cache=True)
            out += [s.strip() for s in tproc.tokenizer.batch_decode(ids, skip_special_tokens=True)]
        return out

    for k, p in enumerate(todo, 1):
        idx = [i for i, l in enumerate(classes[p['image']]) if l in TEXT_TYPES]
        texts = {}
        if idx:
            with Image.open(p['path']) as im:
                page = im.convert('RGB')
                crops = [trocr_crop(page, dets[p['image']][i]['bbox']) for i in idx]
            for i, t in zip(idx, recognize(crops)):
                t, looped = trim_loop(t); loops += looped; texts[str(i)] = t
        trocr[p['image']] = texts
        if k % 25 == 0 or k == len(todo):
            save_json(TXT_FILE, trocr); print(f'TrOCR {len(trocr)}/{len(RUN_PAGES)} pages')
    del tmodel, tproc; free_gpu()
print('TrOCR done | repetition loops trimmed in this run:', loops)
first = RUN_PAGES[0]['image']
for i, t in list(trocr[first].items())[:5]: print(f'  region {i}: {t}')

TrOCR done | repetition loops trimmed in this run: 0
  region 0: Треба любити свій дім, навіть якщо він
  region 1: РЕБи і генератори, а ще, нашкорити вечора-
  region 2: Росіяни хочуть, щоб ми стали схожими
  region 3: але й розрізняємо їх на слух.
  region 4: Треба жити! Не впівсили, не трошки, не завтра,


## 7. Stage 4 — formulas and tables (Qwen3-VL-8B + LoRA, team module unchanged)

In [7]:
FT_FILE = OUT / 'formula_table.json'
ft = load_json(FT_FILE, {})
jobs = [(p, i, l) for p in RUN_PAGES for i, l in enumerate(classes[p['image']])
        if l in ('formula', 'table') and f"{p['image']}#{i}" not in ft]
print('formula/table regions to transcribe:', len(jobs))
if jobs:
    spec = importlib.util.spec_from_file_location('rk_ocr', str(OCR_INFER))
    rk_ocr = importlib.util.module_from_spec(spec); spec.loader.exec_module(rk_ocr)
    QUANT4 = VRAM < 40          # the team's note: bf16 8B may not fit on L4 -> 4-bit
    ocr = rk_ocr.FormulaTableOCR(str(OCR_ADAPTER), quant4=QUANT4)
    print('8B loaded, quant4 =', QUANT4)
    CHUNK = 16
    for s in range(0, len(jobs), CHUNK):
        chunk = jobs[s:s + CHUNK]; crops = []
        for p, i, l in chunk:
            with Image.open(p['path']) as im:
                page = im.convert('RGB')
                box = ocr.expand_box(dets[p['image']][i]['bbox'], *page.size)   # recommended +4% (handoff.json)
                crops.append(page.crop(tuple(int(round(v)) for v in box)))
        texts = ocr.transcribe(crops, [l for _, _, l in chunk], batch_size=4)
        for (p, i, l), t in zip(chunk, texts): ft[f"{p['image']}#{i}"] = t
        save_json(FT_FILE, ft); print(f'formula/table {min(s + CHUNK, len(jobs))}/{len(jobs)}')
    del ocr; free_gpu()
shown = [k for k in ft if k.split('#')[0] in {p['image'] for p in RUN_PAGES}][:5]
for k in shown: print(f'  {k}: {ft[k][:120]!r}')

formula/table regions to transcribe: 0
  8d61156d-be9d-43a5-82b8-8ac68bb0c74f.jpg#8: 'xe з нею ще в чиєму серці поселиться любов до слів:'
  e0e8c060-88c6-443d-90b3-79b1aa3da848.jpg#21: 'N=2'
  26b25869-e0e6-486e-a7bd-ebe8433cc1ae.jpg#20: 'N=4'
  8f883660-a534-4b7a-aca8-18b7fc97bf90.jpg#11: '7. III. 28.'
  455e22e3-0ac0-492f-a969-35c5bdce3853.jpg#0: '23. \\lim_{n \\to \\infty} \\frac{(2n^2 + 1)(n + 3)^3}{(n^2 + 1)^3 + (3n - 1)^2} = \\lim_{n \\to \\infty} \\frac{2n^5 + 18n^4 + '


## 8. Assemble, validate and write the submission CSV

In [9]:
VALID = {'handwritten', 'printed', 'formula', 'table', 'annotation', 'image', 'graph'}
results, invalid = {}, 0
for p in RUN_PAGES:
    n = p['image']; regs = []
    for i, (b, l) in enumerate(zip(dets[n], classes[n])):
        if l == '__invalid__': invalid += 1; l = 'handwritten'
        if l in ('image', 'graph'): text = ''
        elif l in ('formula', 'table'): text = ft[f'{n}#{i}']
        else: text = trocr[n][str(i)]
        x1, y1, x2, y2 = b['bbox']
        assert 0 <= x1 < x2 <= p['width'] and 0 <= y1 < y2 <= p['height'], (n, b)
        assert l in VALID and isinstance(text, str)
        regs.append({'bbox': [x1, y1, x2, y2], 'type': l, 'text': text})
    results[n] = regs
print('invalid classifier answers exported as handwritten:', invalid)

if SMOKE:
    for p in RUN_PAGES:
        print(f"\n=== {p['image']}: {len(results[p['image']])} regions ===")
        for r in sorted(results[p['image']], key=lambda r: (r['bbox'][1], r['bbox'][0]))[:25]:
            print(f"{r['type']:11s} | {r['text'][:90]!r}")
    print('\nSMOKE run: no CSV written. If this looks right, set SMOKE = False in cell 2 and run cells 2-9 again.')
else:
    assert list(results) == EXPECTED
    SUB = OUT / 'submission.csv'
    with open(SUB, 'w', encoding='utf-8', newline='') as f:
        wr = csv.DictWriter(f, fieldnames=['image', 'regions']); wr.writeheader()
        for n in EXPECTED: wr.writerow({'image': n, 'regions': json.dumps(results[n], ensure_ascii=False)})
    with open(SUB, encoding='utf-8', newline='') as f:            # re-read check
        rows = list(csv.DictReader(f))
    assert [r['image'] for r in rows] == EXPECTED and all(isinstance(json.loads(r['regions']), list) for r in rows)
    n_reg = sum(len(v) for v in results.values()); n_empty = sum(1 for v in results.values() for r in v
                                                               if r['type'] not in ('image', 'graph') and not r['text'])
    print(f'CSV OK: {len(rows)} pages, {n_reg} regions, {n_empty} text regions with empty text -> {SUB}')
    import shutil; shutil.copy(SUB, '/content/submission.csv')
    from google.colab import files; files.download('/content/submission.csv')

invalid classifier answers exported as handwritten: 0
CSV OK: 385 pages, 8961 regions, 0 text regions with empty text -> /content/drive/MyDrive/rukopys_submission_v1/full/submission.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Approximate local score on the HF test labels (sanity check only)
This re-implements the published formula: detection F1 at IoU ≥ 0.5, class accuracy on matched pairs, region CER on matched scorable regions, Page CER on reading-order concatenation. The normalization is the team's approximate re-implementation from notebook 11. **It is not the official scorer**, so expect differences from Kaggle. Never use it to tune anything.

In [8]:
from rapidfuzz.distance import Levenshtein
_L = {'alpha':'α','beta':'β','gamma':'γ','delta':'δ','epsilon':'ε','varepsilon':'ε','theta':'θ','lambda':'λ','mu':'μ','pi':'π',
      'rho':'ρ','sigma':'σ','tau':'τ','phi':'φ','varphi':'φ','omega':'ω','Delta':'Δ','Omega':'Ω','cdot':'·','times':'×','div':'÷',
      'pm':'±','leq':'≤','le':'≤','geq':'≥','ge':'≥','neq':'≠','ne':'≠','approx':'≈','infty':'∞','rightarrow':'→','to':'→',
      'leftarrow':'←','Rightarrow':'⇒','in':'∈','sqrt':'√','sum':'∑','int':'∫','degree':'°','angle':'∠','perp':'⊥','ldots':'…','dots':'…'}
_SUP = str.maketrans({'⁰':'^0','¹':'^1','²':'^2','³':'^3','⁴':'^4','⁵':'^5','⁶':'^6','⁷':'^7','⁸':'^8','⁹':'^9','⁺':'^+','⁻':'^-','ⁿ':'^n'})
_SUB = str.maketrans({'₀':'_0','₁':'_1','₂':'_2','₃':'_3','₄':'_4','₅':'_5','₆':'_6','₇':'_7','₈':'_8','₉':'_9'})
def norm(s):
    s = re.sub(r'~~(.*?)~~\{(.*?)\}', r'\2', str(s), flags=re.S); s = re.sub(r'~~(.*?)~~', r'\1', s, flags=re.S)
    s = re.sub(r'\\([A-Za-z]+)', lambda m: _L.get(m.group(1), m.group(0)), s)
    s = s.translate(str.maketrans({'c': 'с', 'o': 'о', 'p': 'р', 'x': 'х'}))
    s = re.sub(r'[–—−]', '-', s); s = re.sub(r'[«»“”„]', '"', s); s = re.sub(r'[‘’‚`]', "'", s)
    s = s.translate(_SUP).translate(_SUB); s = re.sub(r'([_^])\{([^{}\s]+)\}', r'\1\2', s)
    return re.sub(r'\s+', ' ', s).strip()
def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1])); i = ix * iy
    return i / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - i + 1e-9)
scorable = lambda g: g.get('language', 'uk') == 'uk' and g.get('legibility', 'legible') == 'legible' and g['type'] not in ('image', 'graph')

TP = FP = FN = 0; cls_ok = cls_n = 0; reg_cer = []; page_cer = []
for p in RUN_PAGES:
    gt = meta[p['image']].get('regions') or []; pr = results[p['image']]
    pairs = sorted(((iou(g['bbox'], r['bbox']), gi, ri) for gi, g in enumerate(gt) for ri, r in enumerate(pr)), reverse=True)
    ug, ur, match = set(), set(), []
    for v, gi, ri in pairs:
        if v < 0.5: break
        if gi not in ug and ri not in ur: ug.add(gi); ur.add(ri); match.append((gi, ri))
    TP += len(match); FP += len(pr) - len(match); FN += len(gt) - len(match)
    for gi, ri in match:
        cls_n += 1; cls_ok += gt[gi]['type'] == pr[ri]['type']
        if scorable(gt[gi]):
            ref = norm(gt[gi].get('text') or '')
            if ref: reg_cer.append(Levenshtein.distance(ref, norm(pr[ri]['text'])) / len(ref))
    order = lambda rs: sorted(rs, key=lambda r: (r['bbox'][1], r['bbox'][0]))
    gref = norm(' '.join(g.get('text') or '' for g in order([g for g in gt if scorable(g)])))
    gpred = norm(' '.join(r['text'] for r in order(pr)))
    if gref: page_cer.append(Levenshtein.distance(gref, gpred) / len(gref))
if not page_cer:
    print('No HF labels for these pages - approximate score skipped.')
else:
    P, R = TP / max(1, TP + FP), TP / max(1, TP + FN); F1 = 2 * P * R / max(1e-9, P + R)
    acc = cls_ok / max(1, cls_n); cer = sum(reg_cer) / max(1, len(reg_cer)); pcer = sum(page_cer) / len(page_cer)
    score = 0.15 * F1 + 0.05 * acc + 0.30 * (1 - cer) + 0.50 * (1 - pcer)
    print(f'Detection F1 {F1:.4f} (P {P:.3f}, R {R:.3f}) | ClassAcc {acc:.4f} | region CER {cer:.4f} | Page CER {pcer:.4f}')
    print(f'APPROXIMATE score on {len(page_cer)} pages: {score:.4f}   (previous Kaggle submission: 0.57763)')

Detection F1 0.8838 (P 0.851, R 0.920) | ClassAcc 0.8152 | region CER 0.1485 | Page CER 0.3669
APPROXIMATE score on 385 pages: 0.7453   (previous Kaggle submission: 0.57763)


In [10]:
from huggingface_hub import hf_hub_download
new_sample = hf_hub_download('UkrainianCatholicUniversity/rukopys', 'sample_submission.csv',
                             repo_type='dataset')          # latest version (v3.0, 382 pages)
with open(new_sample, encoding='utf-8-sig', newline='') as f:
    NEW = [r['image'] for r in csv.DictReader(f)]
print(len(NEW), 'images expected now')
missing = [n for n in NEW if n not in results]
assert not missing, f'{len(missing)} names not in our predictions, e.g. {missing[:3]}'
print('left out:', sorted(set(results) - set(NEW)))
SUB2 = OUT / 'submission_382.csv'
with open(SUB2, 'w', encoding='utf-8', newline='') as f:
    wr = csv.DictWriter(f, fieldnames=['image', 'regions']); wr.writeheader()
    for n in NEW: wr.writerow({'image': n, 'regions': json.dumps(results[n], ensure_ascii=False)})
import shutil; shutil.copy(SUB2, '/content/submission_382.csv')
from google.colab import files; files.download('/content/submission_382.csv')

382 images expected now
left out: ['3e920349-b25e-5705-8d5c-839aed38f6c2.jpg', 'b8a1dbdf-e4e5-51a1-8948-b972dd0385b5.jpg', 'f9051dc0-bd73-5f7a-a09a-48b29e41db80.jpg']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Score: 0.81322
Private score: 0.81322